In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/fno_benchmark_data_raw/FO Benchmark_Data")
DB_PATH  = Path("../data/erp_benchmark.db")

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR.resolve()} — unzip FO Benchmark_Data.zip first"

# Setup: Mock MCP Server (CSV → SQLite)

Builds a local SQLite database from the synthetic ERP benchmark data and configures `mcp-server-sqlite` as a drop-in backend for the ERP QA pipeline — no Dynamics 365 licence required.

**Prerequisite:** Unzip the benchmark data archive before running this notebook:

```
data/fno_benchmark_data_raw/FO Benchmark_Data.zip  →  data/fno_benchmark_data_raw/FO Benchmark_Data/
```

See [docs/setting_up_mock_mcp_server.md](../docs/setting_up_mock_mcp_server.md) for the full setup guide including `.mcp.json` and `config.yaml` configuration.

In [ ]:
# Step 1: load master data (customers and vendors)
customers = pd.read_csv(DATA_DIR / "Master Data" / "D365_Customers_DMF_Final.csv")
vendors   = pd.read_csv(DATA_DIR / "Master Data" / "D365_Vendors_DMF_Final.csv")

con = sqlite3.connect(DB_PATH)
customers.to_sql("customers", con, if_exists="replace", index=False)
vendors.to_sql("vendors",   con, if_exists="replace", index=False)
con.close()
print(f"customers: {len(customers):,} rows")
print(f"vendors:   {len(vendors):,} rows")

In [ ]:
# Step 2: load transactional data (AP/AR journals)
# AP file is xlsx (zip), AR file is legacy xls (OLE2) despite the .xlsx extension
ap = pd.read_excel(DATA_DIR / "Transactional Data" / "AP_Journal_Import_Updated.xlsx", engine="openpyxl")
ar = pd.read_excel(DATA_DIR / "Transactional Data" / "AR_Journal_Import_Updated_public.xlsx", engine="openpyxl")

con = sqlite3.connect(DB_PATH)
ap.to_sql("ap_journals", con, if_exists="replace", index=False)
ar.to_sql("ar_journals", con, if_exists="replace", index=False)
con.close()
print(f"ap_journals: {len(ap):,} rows")
print(f"ar_journals: {len(ar):,} rows")

In [ ]:
# Step 3: verify
con = sqlite3.connect(DB_PATH)
for name, count in con.execute(
    "SELECT name, (SELECT count(*) FROM sqlite_master WHERE type='table' AND name=m.name) "
    "FROM sqlite_master m WHERE type='table'"
):
    n = con.execute(f"SELECT count(*) FROM \"{name}\"").fetchone()[0]
    print(f"{name}: {n:,} rows")
con.close()